In [3]:
# GPU-Accelerated Huffman Encoding for PDF Text Compression
# This notebook implements Huffman encoding using CUDA for text compression
!pip install cuda-python
!pip install numpy==1.24.0
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install -q --system numba-cuda==0.4.0
import os
os.environ["NUMBA_CUDA_FORCE_PTX_VERSION"] = "70"
!pip install PyPDF2
!pip install flask
!pip install flask_cors
!pip install pyngrok
!pip install flask_ngrok
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1
from numba import cuda, jit
import math
import pickle
import time
import os
import io
from heapq import heappush, heappop, heapify
from collections import Counter, defaultdict
import PyPDF2
from google.colab import files
import base64

# Check if GPU is available
print("GPU available:", cuda.is_available())
if cuda.is_available():
    device = cuda.get_current_device()
    print(f"Device: {device.name}")
    print(f"Compute Capability: {device.compute_capability}")
    print(f"Max threads per block: {device.MAX_THREADS_PER_BLOCK}")

# Define the Huffman Node class
class HuffmanNode:
    def __init__(self, char, freq):
        self.char = char
        self.freq = freq
        self.left = None
        self.right = None

    def __lt__(self, other):
        return self.freq < other.freq

# Function to build the Huffman tree and generate codes
def build_huffman_tree(text):
    # Count the frequency of each character
    frequency = Counter(text)

    # Create a priority queue
    priority_queue = []
    for char, freq in frequency.items():
        heappush(priority_queue, HuffmanNode(char, freq))

    # Build the Huffman tree
    while len(priority_queue) > 1:
        left = heappop(priority_queue)
        right = heappop(priority_queue)

        # Create a new internal node with these two nodes as children
        # and with frequency equal to the sum of the two nodes' frequencies
        internal_node = HuffmanNode(None, left.freq + right.freq)
        internal_node.left = left
        internal_node.right = right

        heappush(priority_queue, internal_node)

    # The remaining node is the root node
    root = priority_queue[0]

    # Generate Huffman codes
    codes = {}
    def generate_codes(node, code):
        if node:
            if node.char:
                codes[node.char] = code
            generate_codes(node.left, code + "0")
            generate_codes(node.right, code + "1")

    generate_codes(root, "")
    return root, codes

# Function to encode text using Huffman coding
def huffman_encode(text, codes):
    encoded_text = ""
    for char in text:
        encoded_text += codes[char]

    # Pad the encoded text to make its length a multiple of 8
    padding = 8 - (len(encoded_text) % 8)
    if padding < 8:
        encoded_text += "0" * padding

    # Convert binary string to bytes
    encoded_bytes = bytearray()
    for i in range(0, len(encoded_text), 8):
        byte = encoded_text[i:i+8]
        encoded_bytes.append(int(byte, 2))

    return bytes(encoded_bytes), padding

# Function to decode Huffman encoded text
def huffman_decode(encoded_bytes, padding, root):
    # Convert bytes to binary string
    encoded_text = ""
    for byte in encoded_bytes:
        encoded_text += format(byte, '08b')

    # Remove padding
    encoded_text = encoded_text[:-padding] if padding < 8 else encoded_text

    # Decode the text
    decoded_text = ""
    current_node = root
    for bit in encoded_text:
        if bit == '0':
            current_node = current_node.left
        else:
            current_node = current_node.right

        if current_node.char:
            decoded_text += current_node.char
            current_node = root

    return decoded_text

# CUDA kernel for parallel character frequency counting
@cuda.jit
def count_frequency_kernel(text_array, freq_array):
    idx = cuda.grid(1)
    if idx < len(text_array):
        char_code = text_array[idx]
        cuda.atomic.add(freq_array, char_code, 1)

# CPU function to prepare and launch the GPU kernel
def gpu_count_frequency(text):
    # Convert text to array of ASCII codes
    text_array = np.array([ord(c) for c in text], dtype=np.int32)

    # Create frequency array (for all possible ASCII values)
    freq_array = np.zeros(256, dtype=np.int32)

    # Calculate grid and block dimensions
    threads_per_block = 256
    blocks_per_grid = (len(text_array) + threads_per_block - 1) // threads_per_block

    # Launch the kernel
    count_frequency_kernel[blocks_per_grid, threads_per_block](text_array, freq_array)

    # Convert the frequency array to a Counter-like dictionary
    frequency = {chr(i): freq for i, freq in enumerate(freq_array) if freq > 0}
    return frequency

# Function to extract text from PDF
def extract_text_from_pdf(pdf_file):
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page_num in range(len(pdf_reader.pages)):
        page = pdf_reader.pages[page_num]
        text += page.extract_text()
    return text

# Function to compress PDF using Huffman encoding
def compress_pdf_text(text):
    start_time = time.time()

    # Step 1: Count character frequencies using GPU
    frequency = gpu_count_frequency(text)
    freq_time = time.time()

    # Create a priority queue for Huffman tree construction
    priority_queue = []
    for char, freq in frequency.items():
        heappush(priority_queue, HuffmanNode(char, freq))

    # Step 2: Build Huffman tree and generate codes (CPU task)
    root, codes = build_huffman_tree(text)
    tree_time = time.time()

    # Step 3: Encode the text
    encoded_bytes, padding = huffman_encode(text, codes)
    encode_time = time.time()

    # Calculate compression metrics
    original_size = len(text.encode('utf-8'))
    compressed_size = len(encoded_bytes)
    compression_ratio = original_size / compressed_size if compressed_size > 0 else 0

    # Timing information
    timings = {
        'frequency_counting': freq_time - start_time,
        'huffman_tree_building': tree_time - freq_time,
        'text_encoding': encode_time - tree_time,
        'total_time': encode_time - start_time
    }

    # Create compression result
    result = {
        'original_size': original_size,
        'compressed_size': compressed_size,
        'compression_ratio': compression_ratio,
        'encoded_data': encoded_bytes,
        'padding': padding,
        'huffman_codes': codes,
        'timings': timings
    }

    return result, root

# Function to decompress text
def decompress_text(encoded_bytes, padding, root):
    start_time = time.time()
    decoded_text = huffman_decode(encoded_bytes, padding, root)
    end_time = time.time()

    return decoded_text, end_time - start_time

# Function to serialize Huffman tree for storage/transmission
def serialize_huffman_tree(root):
    if root is None:
        return None

    def traverse(node):
        if node.char:  # Leaf node
            return {'type': 'leaf', 'char': node.char, 'freq': node.freq}
        else:  # Internal node
            return {
                'type': 'internal',
                'freq': node.freq,
                'left': traverse(node.left),
                'right': traverse(node.right)
            }

    return traverse(root)

# Function to deserialize Huffman tree
def deserialize_huffman_tree(tree_dict):
    if tree_dict is None:
        return None

    def build_tree(node_dict):
        if node_dict['type'] == 'leaf':
            return HuffmanNode(node_dict['char'], node_dict['freq'])
        else:
            node = HuffmanNode(None, node_dict['freq'])
            node.left = build_tree(node_dict['left'])
            node.right = build_tree(node_dict['right'])
            return node

    return build_tree(tree_dict)

# Function to compress a PDF file
def compress_pdf_file(pdf_file):
    # Extract text from PDF
    text = extract_text_from_pdf(pdf_file)

    # Compress the text
    compression_result, huffman_tree = compress_pdf_text(text)

    # Serialize the Huffman tree for storage
    serialized_tree = serialize_huffman_tree(huffman_tree)

    # Create a compressed file object
    compressed_data = {
        'encoded_data': compression_result['encoded_data'],
        'padding': compression_result['padding'],
        'huffman_tree': serialized_tree,
        'original_size': compression_result['original_size'],
        'compressed_size': compression_result['compressed_size'],
        'compression_ratio': compression_result['compression_ratio']
    }

    return compressed_data

# Function to decompress a compressed PDF file
def decompress_pdf_file(compressed_data):
    # Extract compressed data
    encoded_data = compressed_data['encoded_data']
    padding = compressed_data['padding']
    serialized_tree = compressed_data['huffman_tree']

    # Deserialize the Huffman tree
    huffman_tree = deserialize_huffman_tree(serialized_tree)

    # Decompress the text
    decoded_text, decompression_time = decompress_text(encoded_data, padding, huffman_tree)

    return decoded_text, decompression_time

# Function to save compressed data to a file
def save_compressed_data(compressed_data, filename):
    with open(filename, 'wb') as f:
        pickle.dump(compressed_data, f)

# Function to load compressed data from a file
def load_compressed_data(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)

# Function to encode compressed data as base64 for transmission to frontend
def encode_compressed_data_base64(compressed_data):
    # Create a bytes buffer
    buffer = io.BytesIO()

    # Pickle the compressed data to the buffer
    pickle.dump(compressed_data, buffer)

    # Get the bytes from the buffer
    bytes_data = buffer.getvalue()

    # Encode as base64
    base64_encoded = base64.b64encode(bytes_data).decode('utf-8')

    return base64_encoded

# Function to decode base64 encoded compressed data
def decode_compressed_data_base64(base64_encoded):
    # Decode base64 to bytes
    bytes_data = base64.b64decode(base64_encoded)

    # Create a bytes buffer
    buffer = io.BytesIO(bytes_data)

    # Unpickle the compressed data from the buffer
    compressed_data = pickle.load(buffer)

    return compressed_data

# Create Flask API for the compression service
from flask import Flask, request, jsonify
from flask_cors import CORS
import base64

app = Flask(__name__)
CORS(app)  # Enable CORS for frontend connections

@app.route('/', methods=['GET'])
def index():
    return jsonify({
        'status': 'running',
        'info': 'PDF Compression API using Huffman Encoding with GPU Acceleration',
        'endpoints': {
            'POST /compress': 'Compress a PDF file',
            'POST /decompress': 'Decompress a previously compressed PDF'
        }
    })

@app.route('/compress', methods=['POST'])
def compress_endpoint():
    if 'file' not in request.files:
        return jsonify({'error': 'No file part'}), 400

    file = request.files['file']
    if file.filename == '':
        return jsonify({'error': 'No selected file'}), 400

    if not file.filename.lower().endswith('.pdf'):
        return jsonify({'error': 'File is not a PDF'}), 400

    try:
        # Compress the PDF
        compressed_data = compress_pdf_file(file)

        # Encode for transmission
        base64_encoded = encode_compressed_data_base64(compressed_data)

        # Return the compressed data and statistics
        return jsonify({
            'success': True,
            'original_size': compressed_data['original_size'],
            'compressed_size': compressed_data['compressed_size'],
            'compression_ratio': compressed_data['compression_ratio'],
            'compressed_data': base64_encoded
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

@app.route('/decompress', methods=['POST'])
def decompress_endpoint():
    data = request.json
    if not data or 'compressed_data' not in data:
        return jsonify({'error': 'No compressed data provided'}), 400

    try:
        # Decode the compressed data
        compressed_data = decode_compressed_data_base64(data['compressed_data'])

        # Decompress the data
        decoded_text, decompression_time = decompress_pdf_file(compressed_data)

        # Return the decompressed text and statistics
        return jsonify({
            'success': True,
            'decompressed_text': decoded_text,
            'decompression_time': decompression_time,
            'original_size': compressed_data['original_size'],
            'compressed_size': compressed_data['compressed_size'],
            'compression_ratio': compressed_data['compression_ratio']
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500

# Install and import ngrok to expose the API to the internet
# First, install pyngrok
!pip install pyngrok -q
from pyngrok import ngrok

# Function to start the ngrok tunnel
def start_ngrok(port):
    # Set your ngrok auth token (if you have one)
    ngrok.set_auth_token("2ny8nXO8BrWtyB2JWD7V168UIYt_3MuBn3FyriZXzVCE231ja")
    # Optional: Uncomment and add your token for longer sessions

    # Start ngrok tunnel to the specified port
    public_url = ngrok.connect(port)
    print(f" * ngrok tunnel available at: {public_url}")
    print(f" * Access the API using: {public_url}/compress or {public_url}/decompress")
    return public_url

# Run the Flask app with ngrok
if __name__ == '__main__':
    # Start ngrok tunnel for the Flask app
    public_url = start_ngrok(5000)

    # Run the Flask app
    app.run(host='0.0.0.0', port=5000)


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --system
GPU available: True
Device: b'Tesla T4'
Compute Capability: (7, 5)
Max threads per block: 1024
 * ngrok tunnel available at: NgrokTunnel: "https://a085-34-16-128-251.ngrok-free.app" -> "http://localhost:5000"
 * Access the API using: NgrokTunnel: "https://a085-34-16-128-251.ngrok-free.app" -> "http://localhost:5000"/compress or NgrokTunnel: "https://a085-34-16-128-251.ngrok-free.app" -> "http://localhost:5000"/decompress
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/dispatcher.py:579: NumbaPerformanceWarning: Grid size 63 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.11/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:890: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
INFO:werkzeug:127.0.0.1 - - [10/Apr/2025 09:11:07] "POST /compress HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2025 09:12:04] "POST /compress HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [10/Apr/2025 09:13:16] "POST /compress/decompress HTTP/1.1" 404 -
INFO:werkzeug:

In [ ]:
from numba import cuda
print(cuda.get_current_device().compute_capability)
